# D3 - US Accidents: Phân lớp + Luật kết hợp

## 1. Nguồn - giấy phép - quy mô

| Mục | Thông tin |
|---|---|
| Dataset | US Accidents |
| Link | https://www.kaggle.com/datasets/sobhanmoosavi/us-accidents |
| Đơn vị công bố | Moosavi et al. (GeoAI), qua Kaggle |
| Giấy phép | CC BY-NC-SA 4.0 |
| Ngày tải | 2024 |
| Quy mô công bố | khoảng 3.0 triệu vụ tai nạn, 48 thuộc tính |
| Kỹ thuật yêu cầu | **Phân lớp + Luật kết hợp** |

**Tri thức lĩnh vực:** dữ liệu mô tả tai nạn giao thông tại Hoa Kỳ theo thời gian, vị trí, thời tiết và hạ tầng. `Severity` là mục tiêu rời rạc phù hợp cho phân lớp; mỗi vụ tai nạn có thể xem như một "giao dịch" gồm các điều kiện đồng thời xảy ra (thời tiết, ngày/đêm, đặc điểm hạ tầng, mức độ nghiêm trọng) — luật kết hợp giúp tìm các tổ hợp điều kiện thường đi cùng nhau và cùng mức độ nghiêm trọng cao.

**Vị trí trong pipeline:** notebook này (Bài 1) khảo sát dữ liệu thô, kiểm tra thiếu/ngoại lệ, trích xuất đặc trưng thời gian và tạo ra bảng chi tiết **đã làm sạch** (`data/processed/D3_us_accidents/accidents_clean.csv`). Bảng này là đầu vào dùng chung cho:
- **Mục 5 (Phân lớp)** trong chính notebook này, qua `accidents_features.csv` (tập đặc trưng số + cờ hạ tầng suy ra từ cùng bảng đã làm sạch).
- **Bài 3 — Luật kết hợp** (`luat-ket-hop-D3.ipynb`, theo đúng định dạng `luat-ket-hop-D2.ipynb`): đọc trực tiếp `accidents_clean.csv`, rồi thực hiện bước biến đổi *đặc thù* của luật kết hợp (rời rạc hóa `start_hour`/`start_weekday`, gộp `Weather_Condition` ít phổ biến, định nghĩa giao dịch/giỏ) — bước này được giải trình trong chính notebook đó.

## 2. Từ điển dữ liệu

| Thuộc tính | Ý nghĩa | Kiểu | Thang đo |
|---|---|---|---|
| `ID` | Mã vụ tai nạn | string | định danh |
| `Severity` | Mức độ nghiêm trọng từ 1 đến 4 | int | thứ hạng |
| `Start_Time`, `End_Time` | Thời điểm bắt đầu/kết thúc | datetime | thời gian |
| `Start_Lat`, `Start_Lng` | Vĩ độ/kinh độ điểm bắt đầu | float | tỷ lệ |
| `Distance(mi)` | Chiều dài ảnh hưởng | float | tỷ lệ |
| `Temperature(F)`, `Visibility(mi)` | Điều kiện thời tiết | float | tỷ lệ |
| `Weather_Condition` | Mô tả thời tiết | category | danh nghĩa |
| `Sunrise_Sunset`, `Day_of_Week` | Bối cảnh ngày/đêm và thứ | category | danh nghĩa |
| `Amenity`, `Bump`, `Crossing`, `Junction` | Đặc điểm hạ tầng xung quanh | bool | nhị phân |

| `start_hour`, `start_month`, `start_weekday` | Giờ/tháng/thứ tách từ `Start_Time` | int | khoảng/danh nghĩa |
| `duration_minutes` | Thời lượng ảnh hưởng = `End_Time` − `Start_Time` (phút) | float | tỷ lệ |

Ma trận kỹ thuật: **D3 = Phân lớp + Luật kết hợp**.

Ghi chú: `start_hour`, `start_month`, `start_weekday`, `duration_minutes` được tách ở notebook này (mục 4) và giữ nguyên dạng số thô — việc rời rạc hóa thành item ngữ cảnh (ví dụ khung giờ, ngày trong tuần/cuối tuần) là bước đặc thù của luật kết hợp, được thực hiện ở Bài 3.

In [1]:
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Tìm thư mục gốc của project
current = Path.cwd()

while current != current.parent:
    if (current / 'data' / 'raw' / 'D3_us_accidents').exists():
        break
    current = current.parent
ROOT = current
RAW = ROOT / 'data' / 'raw' / 'D3_us_accidents'

PROCESSED = ROOT / 'data' / 'processed' / 'D3_us_accidents'
PROCESSED.mkdir(parents=True, exist_ok=True)

OUT = ROOT / 'data' / 'processed' / 'D3_us_accidents' / 'outputs'
OUT.mkdir(parents=True, exist_ok=True)

SAMPLE_ROWS = 200000

def find_file(name):
    matches = list(RAW.rglob(name))

    if not matches:
        raise FileNotFoundError(
            f'Không tìm thấy {name} trong {RAW}'
        )

    return matches[0]

# Đọc dữ liệu: lấy file CSV lớn nhất trong thư mục raw (file tai nạn chính)
csv_files = list(RAW.rglob('US_Accidents_sampled.csv'))
if not csv_files:
    raise FileNotFoundError(f'Không tìm thấy file CSV trong {RAW}')
accident_path = max(csv_files, key=lambda p: p.stat().st_size)

df = pd.read_csv(accident_path, nrows=SAMPLE_ROWS, low_memory=False)
print('File:', accident_path.name, '| kích thước:', df.shape)

File: US_Accidents_sampled.csv | kích thước: (200000, 46)


## 3. Khám phá dữ liệu và giá trị thiếu

Báo cáo thiếu được lưu đầy đủ; các biến số được rà soát bằng quy tắc Z-score. Không xóa ngoại lệ địa lý hay thời tiết một cách máy móc vì tai nạn nghiêm trọng và điều kiện cực đoan là tín hiệu nghiệp vụ.

In [2]:
missing = df.isna().sum().sort_values(ascending=False).rename('missing_count').to_frame()
missing['missing_rate'] = missing['missing_count'] / len(df)
missing.to_csv(OUT / 'missing_report.csv', encoding='utf-8-sig')

numeric_all = df.select_dtypes(include='number').columns
z = ((df[numeric_all] - df[numeric_all].mean()) / df[numeric_all].std()).abs()
outliers = pd.DataFrame({'attribute': numeric_all, 'count_z_gt_3': (z > 3).sum().values})
outliers.to_csv(OUT / 'outlier_report.csv', index=False, encoding='utf-8-sig')

display(missing.head(20))

,missing_count,missing_rate
End_Lng,170158,0.850790
End_Lat,170158,0.850790
Precipitation(in),90316,0.451580
Wind_Chill(F),83715,0.418575
Wind_Speed(mph),20667,0.103335
Visibility(mi),4020,0.020100
Weather_Condition,3951,0.019755
Humidity(%),3795,0.018975
Temperature(F),3530,0.017650
Wind_Direction,3318,0.016590


## 4. Trích xuất đặc trưng thời gian và chuẩn hóa dữ liệu chi tiết

Tách `start_hour`, `start_month`, `start_weekday` và `duration_minutes` từ `Start_Time`/`End_Time`; chuyển các cờ hạ tầng (`Amenity`, `Bump`, `Crossing`, `Junction`, `Railway`, `Stop`, `Traffic_Signal`) từ boolean sang số 0/1.

Hai bảng được xuất ra từ bước này:

- **`accidents_features.csv`** — tập đặc trưng số + cờ hạ tầng (không có `Weather_Condition`/`Sunrise_Sunset`/`ID`), dùng riêng cho mô hình phân lớp ở mục 5.
- **`accidents_clean.csv`** — bảng chi tiết đầy đủ hơn (thêm `ID`, `Severity`, `Weather_Condition`, `Sunrise_Sunset`), chỉ loại các dòng thiếu `Severity`/`Weather_Condition`/`Sunrise_Sunset` — ba cột dùng làm "neo" để tạo item trong Bài 3. Đây là **dữ liệu processed dùng chung cho Bài 3 — Luật kết hợp**: notebook `luat-ket-hop-D3.ipynb` sẽ đọc trực tiếp file này làm điểm xuất phát, thay vì đọc lại dữ liệu thô hay lặp lại các bước khảo sát/làm sạch đã thực hiện ở notebook này (tương tự cách `luat-ket-hop-D2.ipynb` dùng `order_details_clean.csv` từ Bài 1 của bộ D2).

Lưu ý: `start_hour`/`start_weekday` trong `accidents_clean.csv` vẫn ở dạng số nguyên thô (chưa rời rạc hóa) và `Weather_Condition` vẫn giữ toàn bộ danh mục gốc (chưa gộp nhóm ít phổ biến) — bước rời rạc hóa/gộp nhóm này thuộc về Bài 3 và sẽ được giải trình đầy đủ ở đó.


In [3]:
for col in ['Start_Time', 'End_Time']:

    df[col] = pd.to_datetime(df[col], errors='coerce', format='mixed')

df['start_hour'] = df['Start_Time'].dt.hour
df['start_month'] = df['Start_Time'].dt.month
df['start_weekday'] = df['Start_Time'].dt.dayofweek
df['duration_minutes'] = (df['End_Time'] - df['Start_Time']).dt.total_seconds() / 60

flag_cols = [c for c in ['Amenity', 'Bump', 'Crossing', 'Junction', 'Railway', 'Stop', 'Traffic_Signal'] if c in df]
for col in flag_cols:
    df[col] = df[col].fillna(False).astype(int)

# --- Bảng đặc trưng cho phân lớp (mục 5) ---
features = ['start_hour', 'start_month', 'start_weekday', 'duration_minutes', 'Distance(mi)', 'Temperature(F)', 'Visibility(mi)'] + flag_cols
features = [c for c in features if c in df.columns]

df[features + ['Severity']].to_csv(PROCESSED / 'accidents_features.csv', index=False, encoding='utf-8-sig')
display(df[features + ['Severity']].head())

# --- Bảng chi tiết đã làm sạch, dùng chung cho Bài 3 (luật kết hợp) ---
clean_cols = ['ID', 'Severity', 'Weather_Condition', 'Sunrise_Sunset',
            'start_hour', 'start_month', 'start_weekday', 'duration_minutes',
            'Distance(mi)', 'Temperature(F)', 'Visibility(mi)'] + flag_cols
clean_cols = [c for c in clean_cols if c in df.columns]

accidents_clean = df[clean_cols].copy()

n_before = len(accidents_clean)

accidents_clean = accidents_clean.dropna(subset=['Severity', 'Weather_Condition', 'Sunrise_Sunset'])
n_after = len(accidents_clean)

print(f"\naccidents_clean: {n_before:,} -> {n_after:,} dong "
    f"(loai {n_before - n_after:,} dong thieu Severity/Weather_Condition/Sunrise_Sunset, "
    f"{(n_before - n_after) / n_before:.2%})")

accidents_clean.to_csv(PROCESSED / 'accidents_clean.csv', index=False, encoding='utf-8-sig')

display(accidents_clean.head())


,start_hour,start_month,start_weekday,duration_minutes,Distance(mi),Temperature(F),Visibility(mi),Amenity,Bump,Crossing,Junction,Railway,Stop,Traffic_Signal,Severity
0,16,11,0,44.583333,0.01,62.1,10.0,0,0,0,0,0,0,0,2
1,12,9,2,29.700000,0.00,91.9,10.0,0,0,0,0,0,0,0,3
2,20,9,1,30.000000,0.00,66.2,10.0,0,0,0,0,0,0,0,3
3,10,9,1,30.000000,0.00,64.9,10.0,0,0,0,0,0,0,0,3
4,17,8,5,30.000000,0.00,72.0,10.0,0,0,0,0,0,0,0,3



accidents_clean: 200,000 -> 195,845 dong (loai 4,155 dong thieu Severity/Weather_Condition/Sunrise_Sunset, 2.08%)


,ID,Severity,Weather_Condition,Sunrise_Sunset,start_hour,start_month,start_weekday,duration_minutes,Distance(mi),Temperature(F),Visibility(mi),Amenity,Bump,Crossing,Junction,Railway,Stop,Traffic_Signal
0,A-75728,2,Clear,Day,16,11,0,44.583333,0.01,62.1,10.0,0,0,0,0,0,0,0
1,A-80191,3,Clear,Day,12,9,2,29.700000,0.00,91.9,10.0,0,0,0,0,0,0,0
2,A-19865,3,Clear,Night,20,9,1,30.000000,0.00,66.2,10.0,0,0,0,0,0,0,0
3,A-76706,3,Overcast,Day,10,9,1,30.000000,0.00,64.9,10.0,0,0,0,0,0,0,0
4,A-92998,3,Clear,Day,17,8,5,30.000000,0.00,72.0,10.0,0,0,0,0,0,0,0


## 5. Phân lớp mức độ tai nạn

Dùng mẫu tối đa 300.000 dòng để notebook chạy ổn định trên máy cá nhân. Nhãn `Severity` được giữ nguyên bốn lớp; trọng số lớp giúp giảm thiên lệch do mất cân bằng.

In [4]:
model_df = df.dropna(subset=['Severity']).copy()
if len(model_df) > 300000:
    model_df = model_df.sample(300000, random_state=42)

X = model_df[features]
y = model_df['Severity'].astype(int)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

num = X.select_dtypes(include='number').columns.tolist()
prep = ColumnTransformer([('num', Pipeline([('impute', SimpleImputer(strategy='median')), ('scale', StandardScaler())]), num)], remainder='drop')

models = {
    'Logistic Regression': LogisticRegression(max_iter=300, class_weight='balanced'),
    'Random Forest': RandomForestClassifier(n_estimators=120, random_state=42, n_jobs=-1, class_weight='balanced_subsample')
}

results = []
for name, estimator in models.items():
    pipe = Pipeline([('prep', prep), ('model', estimator)])
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    results.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, pred),
        'Precision': precision_score(y_test, pred, average='weighted', zero_division=0),
        'Recall': recall_score(y_test, pred, average='weighted', zero_division=0),
        'F1': f1_score(y_test, pred, average='weighted', zero_division=0)
    })

results = pd.DataFrame(results)
results.to_csv(OUT / 'classification_results.csv', index=False, encoding='utf-8-sig')
display(results)

,Model,Accuracy,Precision,Recall,F1
0,Logistic Regression,0.358275,0.718012,0.358275,0.398048
1,Random Forest,0.703425,0.677158,0.703425,0.679695


## 6. Luật kết hợp giữa điều kiện tai nạn — thực hiện ở Bài 3

Việc chuẩn bị giao dịch đặc thù cho luật kết hợp (rời rạc hóa `start_hour` thành khung giờ, `start_weekday` thành ngày trong tuần/cuối tuần, gộp `Weather_Condition` ít phổ biến, định nghĩa mỗi vụ tai nạn là một giao dịch và mỗi điều kiện đồng thời xảy ra là một item), thuật toán Apriori/FP-Growth, các độ đo support/confidence/lift và bước lọc luật được thực hiện đầy đủ trong notebook riêng **`luat-ket-hop-D3.ipynb`**, theo đúng định dạng của `luat-ket-hop-D2.ipynb`.

Notebook đó đọc trực tiếp `data/processed/D3_us_accidents/accidents_clean.csv` vừa tạo ở mục 4 làm điểm xuất phát, thay vì đọc lại dữ liệu thô hay lặp lại các bước khảo sát/làm sạch đã thực hiện ở notebook này. Việc tách riêng giúp tránh có hai phân tích luật kết hợp khác nhau (một bản đơn giản ở đây, một bản đầy đủ ở Bài 3) cho cùng một dữ liệu.
